# Egyptian Arabic Speech-to-Text — Whisper Medium + LoRA
**Model:** `openai/whisper-medium` fine-tuned with LoRA (PEFT)   
**Platform:** Kaggle (single T4 GPU — 15GB VRAM)

### All fixes applied
| Fix | Detail |
|---|---|
| Model | `whisper-medium` (769M, fits T4) instead of large-v3 (1.55B, OOM on T4) |
| Single GPU | `CUDA_VISIBLE_DEVICES=0` set **before** `import torch` — fixes DataParallel crash |
| `max_new_tokens` | 225 (was 24 — was truncating every sentence) |
| Normalization | Same `normalize_arabic()` on labels AND predictions |
| `predict_with_generate` | True (was missing — eval WER was wrong) |
| Learning rate | `1e-5` (was `3e-5` — caused catastrophic forgetting) |
| `repetition_penalty` | 1.1 (was 2.0 — broke valid Arabic repetition) |
| dtype | `bf16=True`, `fp16=False` (avoids float/half mismatch on T4) |
| Batch size | `TRAIN_BATCH=8`, `GRAD_ACCUM=4` → effective 32 (was batch=16 → OOM) |
| `generation_max_length` | 225 set in TrainingArguments — matches inference |
| Deduplication | `drop_duplicates(subset=["audio_path"])` before split |
| Preprocessing | `num_proc=4` in dataset.map — 4× faster feature extraction |
| Eval coverage | Full test set evaluated (was capped at 500) |
| Inference helper | Model loaded once outside `transcribe()` — not per-call |
| Batched eval | Inference runs in batches of 16 — ~6× faster than sample-by-sample |


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

import subprocess
pkgs = [
    "transformers>=4.41.0",
    "datasets>=2.19.0",
    "evaluate>=0.4.1",
    "jiwer>=3.0.3",
    "accelerate>=0.34.0",
    "peft>=0.11.0",
    "librosa==0.10.1",
    "soundfile==0.12.1",
    "tensorboard",
    "colorama",
]
for pkg in pkgs:
    result = subprocess.run(["pip", "install", "-q", pkg], capture_output=True)
    if result.returncode != 0:
        print(f"WARNING: failed to install {pkg}")

print("All packages ready.")


In [ ]:
import os, gc, re, json, time, warnings, logging
from pathlib import Path
from typing import List, Tuple
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import tqdm

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

n_gpu = torch.cuda.device_count()
print(f"Visible GPUs : {n_gpu}")
assert n_gpu == 1, f"ERROR: {n_gpu} GPUs visible — restart kernel and rerun from Cell 01"

device = "cuda"
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch      : {torch.__version__}")


## Configuration

In [ ]:
KAGGLE_INPUT   = "/kaggle/input/datasets/fadisarwat/egyptian-arabic-lines"
KAGGLE_WORKING = "/kaggle/working"

class Config:
    # Paths
    DATASET_CSV     = os.path.join(KAGGLE_INPUT,   "index.csv")
    AUDIO_DIR       = os.path.join(KAGGLE_INPUT,   "data")
    CONVERTED_DIR   = os.path.join(KAGGLE_WORKING, "data_wav")
    OUTPUT_DIR      = os.path.join(KAGGLE_WORKING, "training_output")
    MODEL_SAVE_PATH = os.path.join(KAGGLE_WORKING, "whisper-medium-egyptian")
    VIZ_DIR         = os.path.join(KAGGLE_WORKING, "visualizations")
    CHECKPOINT_CSV  = os.path.join(KAGGLE_WORKING, "df_clean_checkpoint.csv")

   
    MODEL_NAME  = "openai/whisper-small"
    LANGUAGE    = "Arabic"
    TASK        = "transcribe"
    SAMPLE_RATE = 16_000

    # ── LoRA ──────────────────────────────────────────────────────────────────
    LORA_R              = 32
    LORA_ALPHA          = 64          
    LORA_DROPOUT        = 0.05
    LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj"]

    # ── Data ──────────────────────────────────────────────────────────────────
    MAX_SAMPLES      = 30_000 #1000000
    MIN_DURATION_SEC = 0.5
    MAX_DURATION_SEC = 30.0
    MIN_TEXT_LEN     = 2
    MAX_TEXT_LEN     = 500

    # ── Training ──────────────────────────────────────────────────────────────
    TEST_SIZE     = 0.1
    SEED          = 42

    TRAIN_BATCH   = 8
    EVAL_BATCH    = 4
    GRAD_ACCUM    = 4        
    LEARNING_RATE = 1e-5    
    WARMUP_STEPS  = 200
    MAX_STEPS     = 4_000
    SAVE_STEPS    = 500
    EVAL_STEPS    = 500
    LOG_STEPS     = 50
    GRAD_CKPT     = True

    PLOT_DPI = 120

cfg = Config()

for d in [cfg.CONVERTED_DIR, cfg.OUTPUT_DIR, cfg.MODEL_SAVE_PATH, cfg.VIZ_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Model         : {cfg.MODEL_NAME}")
print(f"Max steps     : {cfg.MAX_STEPS}")
print(f"LoRA rank     : {cfg.LORA_R}")
print(f"Learning rate : {cfg.LEARNING_RATE}")
print(f"Train batch   : {cfg.TRAIN_BATCH}  (was 16 — OOM risk on T4)")
print(f"Eff. batch    : {cfg.TRAIN_BATCH * cfg.GRAD_ACCUM}")


## Step 1 — Load Dataset

In [ ]:
print("=" * 60)
print("  STEP 1 — Loading Dataset")
print("=" * 60)

df = pd.read_csv(cfg.DATASET_CSV)
df.columns = [c.strip().lower() for c in df.columns]
df["audio_path"] = df["audio_file"].apply(
    lambda x: os.path.join(cfg.AUDIO_DIR, str(x).strip())
)

print(f"  Shape   : {df.shape}")
print(f"  Columns : {df.columns.tolist()}")
print(f"  Sample  :\n{df.head(3).to_string()}")

sample_check = df["audio_path"].head(5).apply(os.path.exists)
print(f"\n  First 5 files exist: {sample_check.tolist()}")

if "gender" in df.columns:
    print(f"\n  Gender distribution:\n{df['gender'].value_counts().to_string()}")

print(f"\n  Total raw samples: {len(df):,}")


## Step 2 — Text Normalization

**Critical fix:** `normalize_arabic()` is applied identically to both training labels
and evaluation predictions. In the original notebook these used different logic,
which inflated WER artificially.


In [ ]:
def normalize_arabic(text: str) -> str:
    """
    Normalize Arabic text consistently.
    MUST be called on both training labels AND evaluation predictions.
    """
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)    # remove tashkeel (diacritics)
    text = re.sub(r"[إأآ]", "ا", text)                       # normalize alef variants
    text = re.sub(r"ؤ", "و", text)                            # normalize waw
    text = re.sub(r"[ىئ]", "ي", text)                         # normalize ya
    text = re.sub(r"ة", "ه", text)                             # normalize tah marbuta
    text = re.sub(r"ـ", "", text)                              # remove tatweel
    text = re.sub(r"[^\u0600-\u06FF\s\d]", " ", text)    # keep Arabic + digits
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Quick tests — including edge cases
test_cases = [
    ("الْقَاهِرَةُ إحدى أكبر المدن", "basic diacritics + alef"),
    ("إزيَّك يا صاحبي؟",             "alef + punctuation"),
    ("أنا بحبّ مصر أوي!",            "alef + tatweel"),
    ("",                              "empty string"),
    (None,                            "None input"),
    ("hello world",                   "latin-only → stripped"),
]
print(f"{'Input':<40} {'Output':<35} {'Case'}")
print("-" * 90)
for inp, desc in test_cases:
    out = normalize_arabic(inp)
    print(f"  {str(inp):<38} {repr(out):<35} {desc}")


## Step 3 — Data Cleaning & MP3→WAV Conversion

In [ ]:
'''
print("=" * 60)
print("  STEP 3 — Data Cleaning + MP3→WAV")
print("=" * 60)

# ── 1. Reload CSV ─────────────────────────────────────────────────────────────
df = pd.read_csv(cfg.DATASET_CSV)
df.columns = [c.strip().lower() for c in df.columns]
df["audio_path"] = df["audio_file"].apply(
    lambda x: os.path.join(cfg.AUDIO_DIR, str(x).strip())
)
print(f"  Loaded {len(df):,} rows")

# ── 2. Normalize text ─────────────────────────────────────────────────────────
print("  Normalizing text...")
df["text"] = df["text"].apply(normalize_arabic)
df["text_length"] = df["text"].apply(len)

# ── 3. Filter ─────────────────────────────────────────────────────────────────
print("  Filtering...")
n0 = len(df)
df = df[df["audio_path"].apply(os.path.exists)].copy()
df = df[(df["text_length"] >= cfg.MIN_TEXT_LEN) &
        (df["text_length"] <= cfg.MAX_TEXT_LEN)].copy()
df = df[df["text"].str.strip().str.len() > 0].copy()

# FIX: deduplicate on audio_path to prevent train/test leakage
before_dedup = len(df)
df = df.drop_duplicates(subset=["audio_path"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"  Removed {before_dedup - len(df):,} duplicates")
print(f"  After filtering: {len(df):,} rows  (removed {n0 - len(df):,} total)")

# ── 4. Stratified subsample ───────────────────────────────────────────────────
if cfg.MAX_SAMPLES and len(df) > cfg.MAX_SAMPLES:
    if "gender" in df.columns:
        stratified = (
            df.groupby("gender", group_keys=False)
            .apply(lambda x: x.sample(
                min(len(x), int(np.ceil(cfg.MAX_SAMPLES * len(x) / len(df)))),
                random_state=cfg.SEED))
            .reset_index(drop=True)
        )
        df = stratified.sample(
            min(cfg.MAX_SAMPLES, len(stratified)),
            random_state=cfg.SEED
        ).reset_index(drop=True)
    else:
        df = df.sample(cfg.MAX_SAMPLES, random_state=cfg.SEED).reset_index(drop=True)
    print(f"  After subsample: {len(df):,} rows")

if "gender" in df.columns:
    print(f"  Gender split: {df['gender'].value_counts().to_dict()}")

# ── 5. MP3 → 16kHz WAV ───────────────────────────────────────────────────────
os.makedirs(cfg.CONVERTED_DIR, exist_ok=True)

def convert_one(args: Tuple[str, str]) -> Tuple[str, bool, float]:
    import librosa, soundfile as sf, os
    mp3_path, wav_path = args
    if os.path.exists(wav_path):
        try:
            info = sf.info(wav_path)
            # FIX: verify file is non-empty before skipping
            if info.duration > 0:
                return wav_path, True, info.duration
        except Exception:
            pass
    try:
        y, _ = librosa.load(mp3_path, sr=16_000, mono=True)
        if len(y) == 0:
            return wav_path, False, -1.0
        sf.write(wav_path, y, 16_000, subtype="PCM_16")
        return wav_path, True, len(y) / 16_000
    except Exception:
        return wav_path, False, -1.0

tasks, wav_paths = [], []
for _, row in df.iterrows():
    wav_name = Path(row["audio_path"]).stem + ".wav"
    wav_path = os.path.join(cfg.CONVERTED_DIR, wav_name)
    wav_paths.append(wav_path)
    tasks.append((row["audio_path"], wav_path))

already_done = sum(1 for _, w in tasks if os.path.exists(w))
to_convert   = [(m, w) for m, w in tasks if not os.path.exists(w)]
print(f"\n  Already converted : {already_done:,}")
print(f"  Need conversion   : {len(to_convert):,}")

if to_convert:
    n_cores = max(1, multiprocessing.cpu_count() - 1)
    print(f"  Using {n_cores} CPU cores...")
    success = 0
    with ProcessPoolExecutor(max_workers=n_cores) as executor:
        futures = {executor.submit(convert_one, t): t for t in to_convert}
        for future in tqdm(as_completed(futures), total=len(futures), desc="MP3→WAV"):
            _, ok, _ = future.result()
            if ok:
                success += 1
    print(f"  Converted: {success:,}/{len(to_convert):,}")

# ── 6. Read durations + filter ───────────────────────────────────────────────
print("\n  Reading durations...")
durations_wav = []
for wav_path in tqdm(wav_paths, desc="Durations"):
    if os.path.exists(wav_path):
        try:
            durations_wav.append(sf.info(wav_path).duration)
        except Exception:
            durations_wav.append(-1.0)
    else:
        durations_wav.append(-1.0)

df["wav_path"]     = wav_paths
df["duration_sec"] = durations_wav

before = len(df)
df = df[
    df["wav_path"].apply(os.path.exists) &
    (df["duration_sec"] >= cfg.MIN_DURATION_SEC) &
    (df["duration_sec"] <= cfg.MAX_DURATION_SEC)
].reset_index(drop=True)
print(f"  Duration filter: removed {before - len(df):,}  | kept {len(df):,}")

total_hours = df["duration_sec"].sum() / 3600

# ── 7. Save checkpoint ────────────────────────────────────────────────────────
df.to_csv(cfg.CHECKPOINT_CSV, index=False)

print("\n" + "=" * 60)
print(f"  Final samples : {len(df):,}")
print(f"  Total audio   : {total_hours:.2f} hours")
print(f"  Checkpoint    : {cfg.CHECKPOINT_CSV}")
print("=" * 60)
'''

## Step 4 — Dataset Visualization

In [ ]:
'''
df = pd.read_csv(cfg.CHECKPOINT_CSV)
print(f"Loaded {len(df):,} rows for visualization")

fig = plt.figure(figsize=(18, 10))
fig.suptitle("Egyptian Arabic Dataset — Overview", fontsize=15, fontweight="bold")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df["duration_sec"].dropna(), bins=50, color="#4C72B0", edgecolor="white", alpha=0.85)
ax1.axvline(cfg.MIN_DURATION_SEC, color="red",    ls="--", lw=1.5, label=f"Min {cfg.MIN_DURATION_SEC}s")
ax1.axvline(cfg.MAX_DURATION_SEC, color="orange", ls="--", lw=1.5, label=f"Max {cfg.MAX_DURATION_SEC}s")
ax1.set_xlabel("Duration (s)"); ax1.set_ylabel("Count")
ax1.set_title("Audio Duration Distribution"); ax1.legend(fontsize=8)

ax2 = fig.add_subplot(gs[0, 1])
text_lens = df["text"].apply(lambda x: len(str(x)))
ax2.hist(text_lens, bins=50, color="#55A868", edgecolor="white", alpha=0.85)
ax2.set_xlabel("Text Length (chars)"); ax2.set_ylabel("Count")
ax2.set_title("Transcription Length Distribution")

ax3 = fig.add_subplot(gs[0, 2])
if "gender" in df.columns:
    gc_vals = df["gender"].value_counts()
    ax3.pie(gc_vals.values, labels=gc_vals.index, autopct="%1.1f%%",
            colors=["#4C72B0","#C44E52","#8172B2"], startangle=90)
    ax3.set_title("Gender Distribution")

ax4 = fig.add_subplot(gs[1, 0:2])
sc = ax4.scatter(df["duration_sec"], text_lens, alpha=0.3, s=6,
                 c=df["duration_sec"], cmap="viridis")
plt.colorbar(sc, ax=ax4, label="Duration (s)")
ax4.set_xlabel("Duration (s)"); ax4.set_ylabel("Text Length (chars)")
ax4.set_title("Duration vs Transcription Length")

ax5 = fig.add_subplot(gs[1, 2])
pcts  = [10, 25, 50, 75, 90, 95, 99]
pvals = np.percentile(df["duration_sec"].dropna(), pcts)
bars  = ax5.barh([f"P{p}" for p in pcts], pvals,
                 color=plt.cm.Blues(np.linspace(0.4, 0.9, len(pcts))), edgecolor="white")
for bar, val in zip(bars, pvals):
    ax5.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}s", va="center", fontsize=8)
ax5.set_xlabel("Duration (s)"); ax5.set_title("Duration Percentiles")

plt.savefig(os.path.join(cfg.VIZ_DIR, "01_dataset_overview.png"),
            dpi=cfg.PLOT_DPI, bbox_inches="tight")
plt.show()
print("Saved: 01_dataset_overview.png")
'''


## Step 5 — Build DatasetDict

Step 6 — Load Processor + Whisper  with LoRA

In [ ]:
import glob
from datasets import Dataset, DatasetDict, concatenate_datasets

# Look at the Input folder now!
train_chunk_dirs = sorted(glob.glob("/kaggle/input/datasets/mariamtobar/egyptian-audio-train-chunks/chunk_*"))
test_chunk_dirs  = sorted(glob.glob("/kaggle/input/datasets/mariamtobar/egyptian-audio-test-chunks/chunk_*"))

print(f"Train chunks: {len(train_chunk_dirs)}")
print(f"Test chunks : {len(test_chunk_dirs)}")

ds = DatasetDict({
    "train": concatenate_datasets([Dataset.load_from_disk(d) for d in train_chunk_dirs]),
    "test":  concatenate_datasets([Dataset.load_from_disk(d) for d in test_chunk_dirs]),
})

ds = ds.with_format("numpy")

print(f"Train : {len(ds['train']):,} samples")
print(f"Test  : {len(ds['test']):,} samples")
print("ds ready.")

In [ ]:
print("=" * 60)
print("  STEP 6 — Load Whisper Small + Apply LoRA")
print("=" * 60)

processor         = WhisperProcessor.from_pretrained(
    cfg.MODEL_NAME, language=cfg.LANGUAGE, task=cfg.TASK
)
feature_extractor = processor.feature_extractor
tokenizer         = processor.tokenizer
print(f"  Processor loaded: {cfg.MODEL_NAME}")

model = WhisperForConditionalGeneration.from_pretrained(
    cfg.MODEL_NAME, torch_dtype=torch.float32,
)
model.generation_config.language           = cfg.LANGUAGE.lower()
model.generation_config.task               = cfg.TASK
model.generation_config.forced_decoder_ids = None
model.config.use_cache                     = False

for param in model.model.encoder.parameters():
    param.requires_grad = False
print("  Encoder frozen for Phase 1")

lora_config = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = cfg.LORA_R,
    lora_alpha     = cfg.LORA_ALPHA,
    target_modules = cfg.LORA_TARGET_MODULES,
    lora_dropout   = cfg.LORA_DROPOUT,
    bias           = "none",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()
print("=" * 60)

## Step 7 — Data Collator

In [ ]:
class WhisperDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features: list) -> dict:
        input_features = []
        for f in features:
            try:
                mel = torch.tensor(f["input_features"].astype("float32"))
            except Exception:
                mel = torch.zeros(80, 3000)
            input_features.append({"input_features": mel})

        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        label_features = [
            {"input_ids": self.processor.tokenizer(
                normalize_arabic(f["text"])
            ).input_ids}
            for f in features
        ]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = WhisperDataCollator(processor)
print("Collator ready.")

## Step 8 — Evaluation Metrics

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_str  = [normalize_arabic(s) for s in pred_str]
    label_str = [normalize_arabic(s) for s in label_str]
    pairs = [(p, r) for p, r in zip(pred_str, label_str) if r.strip()]
    if not pairs:
        return {"wer": 1.0, "cer": 1.0}
    pred_str, label_str = zip(*pairs)
    wer = wer_metric.compute(predictions=list(pred_str), references=list(label_str))
    cer = cer_metric.compute(predictions=list(pred_str), references=list(label_str))
    return {"wer": round(wer, 4), "cer": round(cer, 4)}

print("Metrics ready.")

## Step 9 — Training Arguments & Trainer

In [ ]:
print("=" * 60)
print("  STEP 9 — Training Config (Phase 1)")
print("=" * 60)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = cfg.OUTPUT_DIR,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 2,
    learning_rate               = 1e-5,
    warmup_steps                = 200,
    max_steps                   = 2_000,
    gradient_checkpointing      = True,
    fp16                        = False,
    bf16                        = True,
    eval_strategy               = "steps",
    predict_with_generate       = True,
    generation_max_length       = 225,
    eval_steps                  = 400,
    save_steps                  = 400,
    logging_steps               = 50,
    report_to                   = ["tensorboard"],
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer",
    greater_is_better           = False,
    push_to_hub                 = False,
    dataloader_num_workers      = 4,
    dataloader_pin_memory       = False,
    remove_unused_columns       = False,
    save_total_limit            = 2,
    label_names                 = ["labels"],
)

trainer = Seq2SeqTrainer(
    args            = training_args,
    model           = model,
    train_dataset   = ds["train"],
    eval_dataset    = ds["test"],
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"  Model     : whisper-small (244M params)")
print(f"  Batch     : 16  eff. 32")
print(f"  Max steps : 2,000")
print(f"  Eval steps: 400")
print("=" * 60)

## Step 10 — Phase 1 Training (Encoder Frozen)

Train only the decoder + LoRA adapters first. Faster and safer —
protects Whisper's pretrained acoustic encoder from being overwritten early.


In [ ]:
print("=" * 60)
print("  STEP 10 — Phase 1: Training (encoder frozen)")
print("=" * 60)

torch.cuda.empty_cache()
gc.collect()
print(f"  GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

last_checkpoint = None
if os.path.isdir(cfg.OUTPUT_DIR):
    checkpoints = [
        os.path.join(cfg.OUTPUT_DIR, d)
        for d in os.listdir(cfg.OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        last_checkpoint = sorted(
            checkpoints, key=lambda x: int(x.split("-")[-1])
        )[-1]
        print(f"  Resuming from: {last_checkpoint}")

start        = time.time()
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
elapsed      = time.time() - start

print(f"\n  Phase 1 complete!")
print(f"  Time         : {elapsed/60:.1f} min")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Steps done   : {train_result.global_step}")

PHASE1_PATH = os.path.join(KAGGLE_WORKING, "phase1_checkpoint")
trainer.save_model(PHASE1_PATH)
processor.save_pretrained(PHASE1_PATH)
print(f"  Saved → {PHASE1_PATH}")

with open(os.path.join(cfg.OUTPUT_DIR, "log_phase1.json"), "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)
print("  Log saved.")


In [ ]:
## Step 11 — Phase 2 Training (Encoder Unfrozen)
#
# Continues from Phase 1 (stage3_final in the attached dataset).
# Phase 1 saved a PEFT/LoRA adapter on top of whisper-small.
# Phase 2: merge the adapter into the base weights → unfreeze encoder
# → fine-tune the whole model at a lower LR so Egyptian Arabic
# phonetics are baked in without destroying pretrained features.
#
# ── Disk budget ───────────────────────────────────────────────────────────────
#  Currently: ~13.4 GiB used / 19.5 GiB total  →  ~6.1 GiB free
#  Phase 2 will add:
#    training_output_phase2/  ~1.4 GB per checkpoint × 2 = ~2.8 GB
#    stage4_phase2_final/     ~460 MB (merged model weights only)
#    Overhead / optimizer     ~0.5 GB
#  Total new: ~3.8 GB  →  projected total ~17.2 GiB  (safe, ~2.3 GiB headroom)
#
# ── Runtime estimate (T4 GPU, whisper-small, encoder unfrozen) ────────────────
#  ~27 000 train samples, effective batch 32 → ~844 steps per epoch
#  max_steps = 1 000
#  Training:   1 000 steps × ~2 s/step             ≈  33 min
#  Evaluation: 5 evals (every 200 steps) × ~8 min  ≈  40 min
#  Total:      ≈  1.2 – 1.5 hours  (well within 15 h budget)
# ─────────────────────────────────────────────────────────────────────────────

import os, gc, time, json, glob
import torch
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)
from peft import PeftModel

KAGGLE_WORKING = "/kaggle/working"

print("=" * 60)
print("  STEP 11 — Phase 2: Full-Model Fine-Tuning (Encoder Unfrozen)")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Paths
# ─────────────────────────────────────────────────────────────────────────────
# The dataset attached in Kaggle contains the Phase 1 output.
# stage3_final = best checkpoint saved after Phase 1 training.
PHASE1_ADAPTER_PATH = (
    "/kaggle/input/datasets/youstenawael/phase1-whisper-checkpoints/stage3_final"
)
BASE_MODEL_NAME = "openai/whisper-small"   # same base used in Phase 1
PHASE2_OUTPUT   = os.path.join(KAGGLE_WORKING, "training_output_phase2")
PHASE2_FINAL    = os.path.join(KAGGLE_WORKING, "stage4_phase2_final")
os.makedirs(PHASE2_OUTPUT, exist_ok=True)
os.makedirs(PHASE2_FINAL,  exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 2. Load base model + merge Phase 1 LoRA adapter
#    Phase 1 saved adapter_model.safetensors + adapter_config.json
#    alongside the processor files — so PHASE1_ADAPTER_PATH has both.
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  Loading base model : {BASE_MODEL_NAME}")
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.float32
)

print(f"  Loading LoRA adapter from : {PHASE1_ADAPTER_PATH}")
model = PeftModel.from_pretrained(base_model, PHASE1_ADAPTER_PATH)

print("  Merging LoRA weights into base model (merge_and_unload)...")
model = model.merge_and_unload()          # returns a plain WhisperForConditionalGeneration
print("  Merge complete — model is now a standard (non-PEFT) Whisper.")

# Load the processor that Phase 1 saved (keeps tokenizer config intact)
processor = WhisperProcessor.from_pretrained(PHASE1_ADAPTER_PATH)
print("  Processor loaded from Phase 1 checkpoint.")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Unfreeze encoder
#    Phase 1 kept encoder frozen; Phase 2 lets the full model adapt.
# ─────────────────────────────────────────────────────────────────────────────
for param in model.model.encoder.parameters():
    param.requires_grad = True

model.config.use_cache = False            # required for gradient checkpointing

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Total params    : {total_params/1e6:.1f}M")
print(f"  Trainable params: {trainable_params/1e6:.1f}M  (encoder now unfrozen)")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Training arguments
#    Key choices vs Phase 1:
#      - LR halved (5e-6 vs 1e-5) — encoder is sensitive, lower LR protects it
#      - save_strategy="steps" with save_total_limit=2 — keeps only 2 checkpoints
#        to stay within the ~6 GiB free headroom
#      - load_best_model_at_end=True — auto-restores best WER checkpoint at end
#      - EarlyStoppingCallback(patience=3) — stops if WER stops improving
# ─────────────────────────────────────────────────────────────────────────────
training_args_p2 = Seq2SeqTrainingArguments(
    output_dir                  = PHASE2_OUTPUT,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 2,          # effective batch = 32
    learning_rate               = 5e-6,       # conservative — encoder is fragile
    warmup_steps                = 100,
    max_steps                   = 1_000,
    gradient_checkpointing      = True,
    fp16                        = False,
    bf16                        = True,       # T4 supports bf16
    eval_strategy               = "steps",
    predict_with_generate       = True,
    generation_max_length       = 225,
    eval_steps                  = 200,
    save_strategy               = "steps",
    save_steps                  = 200,        # aligned with eval so best model works
    save_total_limit            = 2,          # keeps only 2 checkpoints → ~2.8 GB
    load_best_model_at_end      = True,       # restore best WER model automatically
    metric_for_best_model       = "wer",
    greater_is_better           = False,
    logging_steps               = 50,
    report_to                   = ["tensorboard"],
    push_to_hub                 = False,
    dataloader_num_workers      = 4,
    dataloader_pin_memory       = False,
    remove_unused_columns       = False,
    label_names                 = ["labels"],
)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Resume support
#    If the cell was interrupted, pick up from the latest checkpoint.
# ─────────────────────────────────────────────────────────────────────────────
last_checkpoint = None
if os.path.isdir(PHASE2_OUTPUT):
    ckpts = sorted(
        glob.glob(os.path.join(PHASE2_OUTPUT, "checkpoint-*")),
        key=lambda x: int(x.split("-")[-1]),
    )
    if ckpts:
        last_checkpoint = ckpts[-1]
        print(f"\n  [Resume] Found checkpoint: {last_checkpoint}")

trainer_p2 = Seq2SeqTrainer(
    args            = training_args_p2,
    model           = model,
    train_dataset   = ds["train"],
    eval_dataset    = ds["test"],
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

# ─────────────────────────────────────────────────────────────────────────────
# 6. Disk check before training
# ─────────────────────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ["du", "-h", "/kaggle/working", "--max-depth=1"],
    capture_output=True, text=True,
)
print("\n  Disk before Phase 2 training:")
print(result.stdout)

torch.cuda.empty_cache()
gc.collect()
print(f"  GPU free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB\n")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Train
# ─────────────────────────────────────────────────────────────────────────────
print("  Starting Phase 2 training...")
start           = time.time()
train_result_p2 = trainer_p2.train(resume_from_checkpoint=last_checkpoint)
elapsed         = time.time() - start

print(f"\n  Phase 2 complete!")
print(f"  Time         : {elapsed/60:.1f} min")
print(f"  Training loss: {train_result_p2.training_loss:.4f}")
print(f"  Steps done   : {train_result_p2.global_step}")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Save final model (best checkpoint is already loaded by load_best_model_at_end)
# ─────────────────────────────────────────────────────────────────────────────
trainer_p2.save_model(PHASE2_FINAL)
processor.save_pretrained(PHASE2_FINAL)
print(f"\n  Final model saved → {PHASE2_FINAL}")

with open(os.path.join(PHASE2_FINAL, "log_phase2.json"), "w") as f:
    json.dump(trainer_p2.state.log_history, f, indent=2)
print("  Training log saved.")

# ─────────────────────────────────────────────────────────────────────────────
# 9. Disk check after training
# ─────────────────────────────────────────────────────────────────────────────
result = subprocess.run(
    ["du", "-h", "/kaggle/working", "--max-depth=1"],
    capture_output=True, text=True,
)
print("\n  Disk after Phase 2 training:")
print(result.stdout)
print("=" * 60)

In [ ]:
import random
import torch
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_PATH = "/kaggle/working/stage4_phase2_final"

print(f"Loading model from {MODEL_PATH}...\n")
test_processor = WhisperProcessor.from_pretrained(MODEL_PATH)
test_model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to("cuda")
test_model.eval()

test_model.generation_config.language = "arabic"
test_model.generation_config.task = "transcribe"

samples = random.sample(list(ds["test"]), 3)

for i, item in enumerate(samples):
    inputs = torch.tensor(item["input_features"]).unsqueeze(0).to("cuda")
    
    with torch.no_grad():
        predicted_ids = test_model.generate(
            inputs,
            language="arabic",       
            task="transcribe",      
            max_new_tokens=225,
            repetition_penalty=1.1,
            num_beams=5 
        )
        
    prediction = test_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    reference = item["text"] 
    
    print(f"--- Audio Sample {i+1} ---")
    print(f"🎙️ ACTUAL (Reference) : {reference}")
    print(f"🤖 MODEL  (Predicted) : {prediction}")
    print("-" * 50)

In [ ]:
'''
print("=" * 60)
print("  STEP 11 — Phase 2: Unfreeze encoder")
print("=" * 60)

for param in model.model.encoder.parameters():
    param.requires_grad = True

trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable params: {trainable_now/1e6:.1f}M")

PHASE2_OUTPUT = os.path.join(KAGGLE_WORKING, "training_output_phase2")
os.makedirs(PHASE2_OUTPUT, exist_ok=True)

training_args_p2 = Seq2SeqTrainingArguments(
    output_dir                  = PHASE2_OUTPUT,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 2,
    learning_rate               = 5e-6,
    warmup_steps                = 100,
    max_steps                   = 1_000,
    gradient_checkpointing      = True,
    fp16                        = False,
    bf16                        = True,
    eval_strategy               = "steps",
    predict_with_generate       = True,
    generation_max_length       = 225,
    eval_steps                  = 200,
    save_steps                  = 200,
    logging_steps               = 50,
    report_to                   = ["tensorboard"],
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer",
    greater_is_better           = False,
    push_to_hub                 = False,
    dataloader_num_workers      = 4,
    dataloader_pin_memory       = False,
    remove_unused_columns       = False,
    save_total_limit            = 2,
    label_names                 = ["labels"],
)

trainer_p2 = Seq2SeqTrainer(
    args            = training_args_p2,
    model           = model,
    train_dataset   = ds["train"],
    eval_dataset    = ds["test"],
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

torch.cuda.empty_cache()
gc.collect()

start           = time.time()
train_result_p2 = trainer_p2.train()
elapsed         = time.time() - start

print(f"\n  Phase 2 complete!")
print(f"  Time         : {elapsed/60:.1f} min")
print(f"  Training loss: {train_result_p2.training_loss:.4f}")
print(f"  Steps done   : {train_result_p2.global_step}")

trainer_p2.save_model(cfg.MODEL_SAVE_PATH)
processor.save_pretrained(cfg.MODEL_SAVE_PATH)
print(f"\n  Final model saved → {cfg.MODEL_SAVE_PATH}")

with open(os.path.join(cfg.OUTPUT_DIR, "log_phase2.json"), "w") as f:
    json.dump(trainer_p2.state.log_history, f, indent=2)
print("  Log saved.")
'''

## Step 12 — Training Curves

In [ ]:
# FIX: reads from saved JSON files — works even after kernel restart
def load_log(path):
    if not os.path.exists(path):
        print(f"  WARNING: log not found at {path}")
        return []
    with open(path) as f:
        return json.load(f)

log1 = load_log(os.path.join(cfg.OUTPUT_DIR, "log_phase1.json"))
log2 = load_log(os.path.join(cfg.OUTPUT_DIR, "log_phase2.json"))
all_log = log1 + log2

if not all_log:
    print("No training logs found. Run Steps 10 and 11 first.")
else:
    train_steps, train_loss, eval_steps, eval_wer, eval_cer = [], [], [], [], []
    for entry in all_log:
        if "loss" in entry and "eval_loss" not in entry:
            train_steps.append(entry["step"])
            train_loss.append(entry["loss"])
        if "eval_wer" in entry:
            eval_steps.append(entry["step"])
            eval_wer.append(entry["eval_wer"])
            eval_cer.append(entry.get("eval_cer", None))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Training Curves — Whisper Medium + LoRA", fontsize=13, fontweight="bold")

    if train_steps:
        axes[0].plot(train_steps, train_loss, color="#4C72B0", lw=1.5)
    axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss"); axes[0].set_title("Training Loss")

    if eval_steps:
        axes[1].plot(eval_steps, eval_wer, color="#C44E52", marker="o", ms=4, lw=1.5)
        axes[1].axhline(0.30, color="green", ls="--", lw=1, label="Target 30%")
        axes[1].axhline(0.20, color="blue",  ls="--", lw=1, label="Target 20%")
        axes[1].set_xlabel("Step"); axes[1].set_ylabel("WER"); axes[1].set_title("Validation WER")
        axes[1].legend(fontsize=8)

    ec_clean = [(e, v) for e, v in zip(eval_steps, eval_cer) if v is not None]
    if ec_clean:
        es, ec = zip(*ec_clean)
        axes[2].plot(es, ec, color="#8172B2", marker="o", ms=4, lw=1.5)
        axes[2].set_xlabel("Step"); axes[2].set_ylabel("CER"); axes[2].set_title("Validation CER")

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.VIZ_DIR, "02_training_curves.png"),
                dpi=cfg.PLOT_DPI, bbox_inches="tight")
    plt.show()
    print("Saved: 02_training_curves.png")

    if eval_wer:
        best_wer = min(eval_wer)
        best_step = eval_steps[eval_wer.index(best_wer)]
        print(f"\n  Best WER : {best_wer:.4f} ({best_wer*100:.1f}%) at step {best_step}")


## Step 13 — Final Evaluation

Key fixes applied here:
- `max_new_tokens=225` (was 24 — was cutting off almost every sentence)
- `repetition_penalty=1.1` (was 2.0 — was breaking valid Arabic words)
- `normalize_arabic()` applied to both prediction and reference
- **FIX:** Full test set evaluated (was capped at 500)
- **FIX:** Batched inference — ~6× faster than sample-by-sample


In [ ]:
print("=" * 60)
print("  STEP 13 — Final Evaluation on Test Set")
print("=" * 60)

# Load saved model fresh from disk
eval_processor = WhisperProcessor.from_pretrained(cfg.MODEL_SAVE_PATH)
eval_model     = WhisperForConditionalGeneration.from_pretrained(cfg.MODEL_SAVE_PATH)
eval_model     = eval_model.to("cuda")
eval_model.eval()

wer_eval = evaluate.load("wer")
cer_eval = evaluate.load("cer")

# FIX: evaluate the full test set, not just 500 samples
NUM_SAMPLES = len(ds["test"])
subset      = ds["test"].select(range(NUM_SAMPLES))
print(f"  Evaluating {NUM_SAMPLES:,} samples (full test set)...\n")

predictions, references = [], []
EVAL_BATCH_SIZE = 16   # FIX: batched inference — much faster than 1-by-1

skipped = 0
audio_batch, ref_batch = [], []

def run_batch(audio_batch, ref_batch):
    """Run inference on a batch and extend predictions/references lists."""
    inputs = eval_processor(
        audio_batch,
        sampling_rate=16_000,
        return_tensors="pt",
        padding=True,
    ).input_features.to("cuda")

    with torch.no_grad():
        pred_ids = eval_model.generate(
            inputs,
            max_new_tokens       = 225,
            repetition_penalty   = 1.1,
            no_repeat_ngram_size = 3,
            length_penalty       = 1.0,
        )

    preds = eval_processor.batch_decode(pred_ids, skip_special_tokens=True)
    for pred, ref in zip(preds, ref_batch):
        pred_norm = normalize_arabic(pred)
        ref_norm  = normalize_arabic(ref)
        if ref_norm.strip():
            predictions.append(pred_norm)
            references.append(ref_norm)

for i, item in enumerate(tqdm(subset, desc="Evaluating")):
    try:
        audio, _ = librosa.load(item["wav_path"], sr=16_000, mono=True)
        audio_batch.append(audio)
        ref_batch.append(item["text"])
    except Exception:
        skipped += 1
        continue

    if len(audio_batch) == EVAL_BATCH_SIZE:
        run_batch(audio_batch, ref_batch)
        audio_batch, ref_batch = [], []

    if i % 200 == 0 and i > 0 and predictions:
        interim_wer = wer_eval.compute(predictions=predictions, references=references)
        print(f"  [{i}/{NUM_SAMPLES}]  Interim WER: {interim_wer*100:.1f}%")

# Flush remaining samples
if audio_batch:
    run_batch(audio_batch, ref_batch)

print(f"  Skipped (bad audio): {skipped:,}")

wer_final = wer_eval.compute(predictions=predictions, references=references)
cer_final = cer_eval.compute(predictions=predictions, references=references)
accuracy  = round((1 - wer_final) * 100, 2)

print("\n" + "=" * 60)
print("  FINAL RESULTS")
print("=" * 60)
print(f"  WER      : {wer_final:.4f}  ({wer_final*100:.1f}%)")
print(f"  CER      : {cer_final:.4f}  ({cer_final*100:.1f}%)")
print(f"  Accuracy : {accuracy:.1f}%")
print(f"  Samples  : {len(predictions):,}")
print("=" * 60)
if wer_final <= 0.20:
    print("  Target achieved: WER ≤ 20%")
elif wer_final <= 0.30:
    print("  Target achieved: WER ≤ 30%")
else:
    print(f"  WER = {wer_final*100:.1f}% — consider more data or increasing MAX_STEPS")


## Step 14 — Inference Helper

In [ ]:

_infer_processor = None
_infer_model     = None

def _load_infer_model(model_path: str = cfg.MODEL_SAVE_PATH):
    """Load processor and model into module-level cache (once only)."""
    global _infer_processor, _infer_model
    if _infer_processor is None or _infer_model is None:
        print(f"  Loading inference model from {model_path}...")
        _infer_processor = WhisperProcessor.from_pretrained(model_path)
        _infer_model     = WhisperForConditionalGeneration.from_pretrained(model_path)
        _infer_model     = _infer_model.to("cuda" if torch.cuda.is_available() else "cpu")
        _infer_model.eval()
        print("  Inference model ready.")
    return _infer_processor, _infer_model


def transcribe(audio_path: str, model_path: str = cfg.MODEL_SAVE_PATH) -> str:
    """
    Transcribe a single Arabic audio file using the fine-tuned model.

    Parameters
    ----------
    audio_path : path to .wav or .mp3 (16kHz mono preferred)
    model_path : path to saved model directory

    Returns
    -------
    Normalized Arabic transcription string

    Notes
    -----
    Model is loaded once on first call and cached for subsequent calls.
    """
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    proc, mdl = _load_infer_model(model_path)
    dev = next(mdl.parameters()).device

    try:
        audio, _ = librosa.load(audio_path, sr=16_000, mono=True)
    except Exception as e:
        raise RuntimeError(f"Could not load audio from {audio_path}: {e}")

    if len(audio) == 0:
        return ""

    inputs = proc(audio, sampling_rate=16_000, return_tensors="pt")
    inputs = inputs.input_features.to(dev)

    with torch.no_grad():
        ids = mdl.generate(
            inputs,
            max_new_tokens       = 225,
            repetition_penalty   = 1.1,
            no_repeat_ngram_size = 3,
            length_penalty       = 1.0,
        )

    text = proc.batch_decode(ids, skip_special_tokens=True)[0]
    return normalize_arabic(text)


def transcribe_batch(audio_paths: List[str],
                     model_path: str = cfg.MODEL_SAVE_PATH,
                     batch_size: int = 8) -> List[str]:
    """
    Transcribe multiple audio files efficiently in batches.

    Parameters
    ----------
    audio_paths : list of paths to .wav or .mp3 files
    model_path  : path to saved model directory
    batch_size  : number of files to process per GPU batch

    Returns
    -------
    List of normalized Arabic transcription strings (same order as input)
    """
    proc, mdl = _load_infer_model(model_path)
    dev = next(mdl.parameters()).device
    results = [""] * len(audio_paths)

    for start in range(0, len(audio_paths), batch_size):
        batch_paths = audio_paths[start:start + batch_size]
        audios, valid_idx = [], []

        for j, path in enumerate(batch_paths):
            try:
                audio, _ = librosa.load(path, sr=16_000, mono=True)
                if len(audio) > 0:
                    audios.append(audio)
                    valid_idx.append(start + j)
            except Exception:
                pass

        if not audios:
            continue

        inputs = proc(
            audios,
            sampling_rate=16_000,
            return_tensors="pt",
            padding=True,
        ).input_features.to(dev)

        with torch.no_grad():
            ids = mdl.generate(
                inputs,
                max_new_tokens       = 225,
                repetition_penalty   = 1.1,
                no_repeat_ngram_size = 3,
                length_penalty       = 1.0,
            )

        preds = proc.batch_decode(ids, skip_special_tokens=True)
        for idx, pred in zip(valid_idx, preds):
            results[idx] = normalize_arabic(pred)

    return results


import random

print("Demo — single-file transcriptions (model loads once):\n")
samples = random.sample(list(ds["test"]), 3)
for i, item in enumerate(samples):
    pred = transcribe(item["wav_path"])
    ref  = normalize_arabic(item["text"])
    print(f"Sample {i+1}")
    print(f"  REF : {ref}")
    print(f"  PRED: {pred}")
    print()


print("Demo — batch transcription (same 3 samples, one GPU call):\n")
paths = [item["wav_path"] for item in samples]
batch_preds = transcribe_batch(paths)
for i, (item, pred) in enumerate(zip(samples, batch_preds)):
    print(f"Sample {i+1}")
    print(f"  REF : {normalize_arabic(item['text'])}")
    print(f"  PRED: {pred}")
    print()


# STREAMLIT

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import librosa
import re
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# 1. Setup the Page
st.set_page_config(page_title="Egyptian Arabic AI", page_icon="🎙️", layout="centered")
st.title("🎙️ Egyptian Arabic Speech-to-Text")
st.markdown("Upload an audio file to test the Whisper LoRA baseline model!")

# 2. Text Normalizer (from your notebook)
def normalize_arabic(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "": return ""
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[إأآ]", "ا", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"[ىئ]", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ـ", "", text)
    text = re.sub(r"[^\u0600-\u06FF\s\d]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 3. Load Model (Cached so it only loads ONCE)
@st.cache_resource
def load_model():
    model_path = "/kaggle/working/stage4_phase2_final" # Update this if you moved the model
    processor = WhisperProcessor.from_pretrained(model_path)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
    model.eval()
    return processor, model, device

with st.spinner("Loading AI Brain... (This takes a few seconds)"):
    processor, model, device = load_model()

# 4. The UI
uploaded_file = st.file_uploader("Upload an Audio File (WAV/MP3)", type=["wav", "mp3", "m4a"])

if uploaded_file is not None:
    # Play the audio in the browser
    st.audio(uploaded_file, format="audio/wav")
    
    if st.button("Transcribe Audio 🚀"):
        with st.spinner("Listening and translating..."):
            try:
                # Load audio from the uploaded file
                audio, _ = librosa.load(uploaded_file, sr=16_000, mono=True)
                
                # Process features
                inputs = processor(audio, sampling_rate=16_000, return_tensors="pt").input_features.to(device)
                
                # Generate text using Beam Search and strict Arabic forcing
                with torch.no_grad():
                    predicted_ids = model.generate(
                        inputs,
                        language="arabic",
                        task="transcribe",
                        max_new_tokens=225,
                        repetition_penalty=1.1,
                        num_beams=5
                    )
                
                # Decode and clean
                prediction = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
                clean_text = normalize_arabic(prediction)
                
                # Display Results
                st.success("Transcription Complete!")
                st.markdown("### 📝 Text:")
                st.info(clean_text)
                
            except Exception as e:
                st.error(f"An error occurred: {e}")

In [ ]:
!pip install -q streamlit
!npm install -g localtunnel

In [ ]:
!wget -q -O - ipv4.icanhazip.com

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
import shutil
import os

# Zip the main outputs
shutil.make_archive('/kaggle/working/stage4_phase2_final', 'zip', '/kaggle/working/stage4_phase2_final')
shutil.make_archive('/kaggle/working/training_output_phase2', 'zip', '/kaggle/working/training_output_phase2')
shutil.make_archive('/kaggle/working/visualizations', 'zip', '/kaggle/working/visualizations')

print("Done! Download these from the Output panel:")
print("- stage4_phase2_final.zip")
print("- training_output_phase2.zip")
print("- visualizations.zip")

In [ ]:
!pip install -q optimum[onnxruntime]

from optimum.exporters.onnx import main_export

main_export(
    model_name_or_path="/kaggle/working/training_output_phase2/checkpoint-1000",
    output="/kaggle/working/whisper_egyptian_onnx",
    task="automatic-speech-recognition",
    framework="pt",
)

# Then zip it
import shutil
shutil.make_archive('/kaggle/working/whisper_egyptian_onnx', 'zip', '/kaggle/working/whisper_egyptian_onnx')
print("Ready to download: whisper_egyptian_onnx.zip")